# Chapter 6: Bayesian Model Comparison

> *"All models are wrong, but some are useful — and Bayes tells you which."*

Given two models, the **Bayes factor** tells us which the data prefer:
$$B_{12} = Z_1 / Z_2$$

This notebook applies model comparison to the detection of spectral emission lines —
one of the most common inference tasks in X-ray and optical astronomy.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
np.random.seed(42)
print("Ready.")


## 6.1 Information Criteria — Quick Model Selection

When nested sampling is too expensive, information criteria approximate ln Z.

| Criterion | Formula | Notes |
|-----------|---------|-------|
| AIC | -2 ln L_max + 2k | Large-sample approx, flat prior |
| AICc | AIC + 2k(k+1)/(N-k-1) | Corrected for small N |
| BIC | -2 ln L_max + k ln N | Stronger penalty for complex models |
| DIC | D_bar + p_D | Uses full posterior, good for hierarchical |

ΔAIC > 10: decisive; 4–7: considerable; < 2: not worth choosing


In [ ]:
# AIC/BIC comparison: polynomial fits to a light curve
np.random.seed(0)
t    = np.linspace(0, 4*np.pi, 40)
y    = 1.5*np.sin(t) + 0.3*np.sin(3*t) + 0.5*np.random.randn(40)
yerr = 0.5*np.ones(40)

degrees  = [1, 2, 3, 5, 8, 12]
aic_vals = []
bic_vals = []
chi2_vals= []

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for idx, deg in enumerate(degrees):
    ax = axes[idx//3, idx%3]

    # Fit polynomial
    X_des = np.vstack([t**i for i in range(deg+1)]).T
    coeffs, res, _, _ = np.linalg.lstsq(X_des/yerr[:, None], y/yerr, rcond=None)
    y_fit    = X_des @ coeffs
    residuals = (y - y_fit) / yerr
    chi2     = np.sum(residuals**2)
    N, k     = len(t), deg+1
    log_like = -0.5*(chi2 + N*np.log(2*np.pi) + 2*np.sum(np.log(yerr)))
    aic      = -2*log_like + 2*k
    aicc     = aic + 2*k*(k+1)/(N-k-1)
    bic      = -2*log_like + k*np.log(N)

    aic_vals.append(aic); bic_vals.append(bic); chi2_vals.append(chi2)

    t_plot = np.linspace(0, 4*np.pi, 300)
    X_plot = np.vstack([t_plot**i for i in range(deg+1)]).T
    y_plot = X_plot @ coeffs

    ax.errorbar(t, y, yerr=yerr, fmt='o', color="#888780", ms=4, elinewidth=0.8)
    ax.plot(t_plot, y_plot, "#3B8BD4", lw=2)
    ax.plot(t_plot, 1.5*np.sin(t_plot)+0.3*np.sin(3*t_plot),
            "#1D9E75", lw=1.5, ls="--", alpha=0.7, label="True")
    ax.set_title(f"Degree {deg}  |  AICc={aicc:.1f}  BIC={bic:.1f}
χ²/dof={chi2/(N-k):.2f}")
    ax.legend(fontsize=8, frameon=False)

plt.suptitle("Polynomial Fits: AIC/BIC Find the Sweet Spot Between Under/Overfitting",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print(f"
{'Degree':>6} {'k':>4} {'chi2/dof':>10} {'AIC':>10} {'BIC':>10}")
for deg, aic, bic, c2 in zip(degrees, aic_vals, bic_vals, chi2_vals):
    k = deg+1
    marker = " ← preferred" if aic == min(aic_vals) else ""
    print(f"{deg:>6} {k:>4} {c2/(len(t)-k):>10.3f} {aic:>10.2f} {bic:>10.2f}{marker}")


## 6.2 Spectral Line Detection — Bayes Factor

**Setup:** We observe an X-ray spectrum. Is there an emission line at ~6.7 keV (Fe XXV)?

- **M₀:** Continuum power-law only
- **M₁:** Continuum + Gaussian emission line

We compute the evidence for each model and the Bayes factor.


In [ ]:
# Simulate an X-ray spectrum with a faint emission line
np.random.seed(17)
E = np.linspace(0.3, 10, 60)
dE = np.diff(E)
E_mid = 0.5*(E[:-1]+E[1:])

# True model parameters
Gamma_true  = 1.9
norm_true   = 0.08
line_E_true = 6.7
line_A_true = 0.006    # faint line
line_s_true = 0.15

mu_true = ((norm_true * E_mid**(-Gamma_true)) +
           line_A_true * stats.norm.pdf(E_mid, line_E_true, line_s_true)) * dE
counts  = np.random.poisson(mu_true)

def log_like_M0(theta):
    # M0: power-law continuum only.
    Gamma, norm = theta
    if norm <= 0: return -np.inf
    mu = norm * E_mid**(-Gamma) * dE
    mu = np.clip(mu, 1e-10, None)
    return np.sum(counts * np.log(mu) - mu)

def log_like_M1(theta):
    # M1: power-law + Gaussian line.
    Gamma, norm, line_A = theta
    if norm <= 0 or line_A < 0: return -np.inf
    mu = (norm * E_mid**(-Gamma) +
          line_A * stats.norm.pdf(E_mid, line_E_true, line_s_true)) * dE
    mu = np.clip(mu, 1e-10, None)
    return np.sum(counts * np.log(mu) - mu)

# Grid over parameter space to estimate evidence numerically
Gamma_grid = np.linspace(1.2, 2.6, 60)
norm_grid  = np.linspace(0.02, 0.18, 60)
lineA_grid = np.linspace(0.0, 0.025, 50)

# Evidence M0
Z0_unnorm = np.zeros((len(Gamma_grid), len(norm_grid)))
for i, G in enumerate(Gamma_grid):
    for j, N in enumerate(norm_grid):
        Z0_unnorm[i,j] = np.exp(log_like_M0([G, N]))
dG = Gamma_grid[1]-Gamma_grid[0]; dN = norm_grid[1]-norm_grid[0]
Z0 = Z0_unnorm.sum() * dG * dN

# Evidence M1 (marginalise over line amplitude too)
Z1_sum = 0
dA = lineA_grid[1]-lineA_grid[0]
for A in lineA_grid:
    for i, G in enumerate(Gamma_grid):
        for j, N in enumerate(norm_grid):
            Z1_sum += np.exp(log_like_M1([G, N, A])) * dG * dN * dA
Z1 = Z1_sum

lnB = np.log(Z1) - np.log(Z0)
print(f"ln Z(M0, no line)  = {np.log(Z0):.2f}")
print(f"ln Z(M1, with line)= {np.log(Z1):.2f}")
print(f"ln Bayes factor    = {lnB:.2f}")

if   lnB > 5:   verdict = "Decisive evidence for the line"
elif lnB > 2.5: verdict = "Strong evidence for the line"
elif lnB > 1.0: verdict = "Substantial evidence for the line"
elif lnB > 0:   verdict = "Weak evidence for the line"
else:           verdict = "Evidence favours no-line model"
print(f"→ {verdict}")


In [ ]:
# Visualise: spectrum + both model predictions + posterior on line amplitude
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Spectrum
axes[0].step(E_mid, counts, where="mid", color="#3B8BD4", lw=1.5, label="Observed counts")
axes[0].plot(E_mid, mu_true/dE, "#D85A30", lw=2, ls="--", label="True model")

# Best-fit M0 (MLE)
idx_M0 = np.unravel_index(Z0_unnorm.argmax(), Z0_unnorm.shape)
G_mle, N_mle = Gamma_grid[idx_M0[0]], norm_grid[idx_M0[1]]
axes[0].plot(E_mid, N_mle*E_mid**(-G_mle), "#1D9E75", lw=2, ls=":",
             label=f"M0 (no line): Γ={G_mle:.1f}")
axes[0].set_xlabel("Energy (keV)"); axes[0].set_ylabel("Counts/keV")
axes[0].set_title("X-ray Spectrum — Is There an Fe Line at 6.7 keV?")
axes[0].axvline(6.7, color="gray", lw=1, ls="--", alpha=0.5)
axes[0].legend(fontsize=8, frameon=False)

# Marginal posterior on line amplitude
post_lineA = np.array([sum(np.exp(log_like_M1([G, N, A]))*dG*dN
                            for G in Gamma_grid for N in norm_grid)
                       for A in lineA_grid])
post_lineA /= np.trapz(post_lineA, lineA_grid)

axes[1].fill_between(lineA_grid, post_lineA, alpha=0.35, color="#7F77DD")
axes[1].plot(lineA_grid, post_lineA, "#7F77DD", lw=2.5)
axes[1].axvline(line_A_true, color="#D85A30", lw=2, ls="--",
                label=f"True A={line_A_true:.3f}")
post_mean = np.trapz(lineA_grid*post_lineA, lineA_grid)
axes[1].axvline(post_mean, color="#1D9E75", lw=2, ls=":",
                label=f"Post mean={post_mean:.4f}")
axes[1].set_xlabel("Line amplitude A"); axes[1].set_ylabel("Posterior density")
axes[1].set_title("Marginal Posterior on Line Amplitude
(after integrating over Γ and normalisation)")
axes[1].legend(frameon=False)

plt.tight_layout()
plt.show()


## 6.3 Posterior Predictive Checks

Before declaring victory, verify the model actually describes the data.
Draw samples from the posterior and simulate new datasets — do they look like the real data?


In [ ]:
# Posterior predictive check for the spectral model
np.random.seed(22)

n_ppc = 1000
ppc_counts = np.zeros((n_ppc, len(E_mid)))
ppc_max    = np.zeros(n_ppc)
ppc_total  = np.zeros(n_ppc)

# Sample from posterior (using grid posterior as discrete distribution)
post_flat = Z0_unnorm.flatten() / Z0_unnorm.sum()
idx_samples = np.random.choice(len(post_flat), n_ppc, p=post_flat)

for s, idx in enumerate(idx_samples):
    i, j = np.unravel_index(idx, Z0_unnorm.shape)
    G_s, N_s = Gamma_grid[i], norm_grid[j]
    mu_s = N_s * E_mid**(-G_s) * dE
    sim  = np.random.poisson(mu_s)
    ppc_counts[s]  = sim
    ppc_max[s]     = sim.max()
    ppc_total[s]   = sim.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# T1: maximum bin count
T_obs_max = counts.max()
axes[0].hist(ppc_max, bins=25, density=True, color="#3B8BD4", alpha=0.7,
             label="Simulated T = max(counts)")
axes[0].axvline(T_obs_max, color="#D85A30", lw=2.5, ls="--",
                label=f"Observed T = {T_obs_max}")
pB1 = np.mean(ppc_max >= T_obs_max)
axes[0].set_title(f"PPC: Maximum Bin Count
p_B = {pB1:.3f}")
axes[0].set_xlabel("Max counts in any bin")
axes[0].legend(frameon=False)

# T2: total counts
T_obs_tot = counts.sum()
axes[1].hist(ppc_total, bins=25, density=True, color="#1D9E75", alpha=0.7,
             label="Simulated T = total counts")
axes[1].axvline(T_obs_tot, color="#D85A30", lw=2.5, ls="--",
                label=f"Observed T = {T_obs_tot}")
pB2 = np.mean(ppc_total >= T_obs_tot)
axes[1].set_title(f"PPC: Total Counts
p_B = {pB2:.3f}")
axes[1].set_xlabel("Total counts")
axes[1].legend(frameon=False)

plt.suptitle("Posterior Predictive Checks (M0: no line)
p_B ≈ 0 or 1 → model fails this test",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print(f"p_B (max count) = {pB1:.3f}  — if < 0.05, model misses the peak")
print(f"p_B (total)     = {pB2:.3f}  — should be ~0.5 if model fits total flux well")


In [ ]:
print("Chapter 6 complete!")
print()
print("Jeffreys scale for Bayes factors:")
print("  ln B < 1.0   : negligible")
print("  ln B 1–2.5   : substantial")
print("  ln B 2.5–5   : strong")
print("  ln B > 5     : decisive")
print()
print("The Bayesian framework automatically implements Occam's razor —")
print("complex models are penalised unless the data genuinely require them.")
